# Retail Customer Segmentation & Product Recommendations

**Author:** Touseeq Danish  
**Methods:** exploratory data analysis (EDA), RFM analysis, K-Means clustering, cluster-level product recommendations.

This notebook uses the reusable functions in `src/pipeline.py`. Download the Online Retail dataset separately (instructions in the repository README); do not commit customer-level data to GitHub.

## 1. Setup and load data
Install dependencies from the repository root with `pip install -r requirements.txt`. Set `DATA_PATH` below to the path of your downloaded `OnlineRetail.xlsx` or CSV file.

In [ ]:
from pathlib import Path
import sys
import pandas as pd
import matplotlib.pyplot as plt

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src").exists() and (PROJECT_ROOT.parent / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

from src.pipeline import (
    load_retail_data, clean_retail_data, calculate_rfm,
    cluster_customers, generate_cluster_recommendations, recommend_for_customer
)

DATA_PATH = PROJECT_ROOT / "data" / "OnlineRetail.xlsx"
if not DATA_PATH.exists():
    raise FileNotFoundError(
        f"Dataset not found at {DATA_PATH}. Download Online Retail and place it in the data/ folder. "
        "See README.md for dataset instructions."
    )

raw = load_retail_data(str(DATA_PATH))
print(f"Rows: {len(raw):,} | Columns: {len(raw.columns)}")
display(raw.head())


## 2. Exploratory data analysis

In [ ]:
display(raw.info())
display(raw.describe(include="all").T)
missing = raw.isna().sum().sort_values(ascending=False)
missing_pct = (missing / len(raw) * 100).round(2)
display(pd.DataFrame({"missing_count": missing, "missing_percent": missing_pct}).query("missing_count > 0"))

if "Country" in raw.columns:
    display(raw["Country"].value_counts().head(10).rename("transactions").to_frame())
if "Description" in raw.columns and "Quantity" in raw.columns:
    qty = raw.groupby("Description", dropna=True)["Quantity"].sum().sort_values(ascending=False).head(10)
    ax = qty.sort_values().plot(kind="barh", figsize=(10, 5), title="Top 10 products by recorded quantity")
    ax.set_xlabel("Quantity")
    plt.tight_layout()
    plt.show()


## 3. Clean transaction data

Rows without a customer ID cannot contribute to customer segmentation, so they are excluded rather than assigned invented IDs. This analysis focuses on positive sales rows; returns and corrections should be analyzed separately if needed.

In [ ]:
clean = clean_retail_data(raw)
print(f"Rows retained: {len(clean):,} / {len(raw):,} ({len(clean) / len(raw):.1%})")
display(clean.head())
display(clean[["Quantity", "UnitPrice", "Revenue"]].describe().T)


## 4. Calculate RFM features
Recency is measured in days relative to one day after the latest transaction date in the dataset. Frequency counts unique invoices and Monetary sums quantity × unit price.

In [ ]:
rfm = calculate_rfm(clean)
display(rfm.head())
display(rfm[["Recency", "Frequency", "Monetary"]].describe().T)


## 5. Segment customers with K-Means
The pipeline applies `log1p` to the non-negative RFM metrics, standardizes them, and fits K-Means. Compare different cluster counts and profile the resulting groups before assigning business labels. Cluster IDs are arbitrary and do not imply a fixed value ranking.

In [ ]:
N_CLUSTERS = min(5, len(rfm))
segmented, kmeans, scaler = cluster_customers(rfm, n_clusters=N_CLUSTERS, random_state=42)
profile = segmented.groupby("Cluster").agg(
    Customers=("CustomerID", "nunique"),
    Median_Recency=("Recency", "median"),
    Median_Frequency=("Frequency", "median"),
    Median_Monetary=("Monetary", "median"),
    Mean_Monetary=("Monetary", "mean"),
).reset_index()
display(profile)
ax = profile.set_index("Cluster")["Customers"].plot(kind="bar", figsize=(8, 4), title="Customers per cluster")
ax.set_ylabel("Customers")
plt.tight_layout()
plt.show()


## 6. Generate cluster-level product recommendations
This transparent baseline ranks products bought by distinct customers within each cluster, using units sold as a tie-breaker. It is not a semantic or causal model, and recommendations can be similar across clusters when buying patterns overlap.

In [ ]:
recommendations = generate_cluster_recommendations(
    clean, segmented, top_n=10, min_customer_count=2
)
display(recommendations.head(30))


## 7. Recommend products for a customer

In [ ]:
customer_id = str(segmented.iloc[0]["CustomerID"])
print("Example customer:", customer_id)
display(segmented[segmented["CustomerID"].astype(str) == customer_id])
display(recommend_for_customer(
    customer_id, segmented, recommendations, transactions=clean, top_n=5
))


## 8. Export results
Generated outputs are excluded from Git by `.gitignore`. Review files locally before sharing, especially if your source data contains personal or commercially sensitive information.

In [ ]:
output_dir = PROJECT_ROOT / "outputs"
output_dir.mkdir(exist_ok=True)
clean.to_csv(output_dir / "cleaned_transactions.csv", index=False)
segmented.to_csv(output_dir / "customer_segments.csv", index=False)
recommendations.to_csv(output_dir / "cluster_product_recommendations.csv", index=False)
print("Saved cleaned_transactions.csv, customer_segments.csv, and cluster_product_recommendations.csv to", output_dir)
